TODO add sklearn models to confirm FairDream properties (and eliminate mlp / add sample weights in fit() method?)

https://scikit-learn.org/stable/supervised_learning.html

In [ ]:
%load_ext autoreload
%autoreload 2


import warnings
warnings.filterwarnings('ignore')

# Discrimination detection and mitigation (on revenues classification dataset)

In [ ]:
import sys
sys.path.append("../")

from fairdream.data_preparation import *
from fairdream.compute_scores import *
from fairdream.detection import *
from fairdream.correction import *
from fairdream.plots import *

from fairdream.experiments import compare_fairness_methods

In [ ]:
# set your statistics purposes, as the type of machine-learning model to train (tree- / neural- based)
model_task = 'classification'
stat_criteria = 'auc'

model_type = 'random_forest' # 'log_reg' # 'neural_net' # 'xgboost'

In [ ]:
# preparing the dataset on clients for binary classification
from sklearn.datasets import fetch_openml
data = fetch_openml(data_id=1590, as_frame=True)

X = data.data
Y = (data.target == '>50K') * 1

## Detection alert on a model trained regardless of fairness

In [ ]:
X_train, X_valid, X_train_valid, X_test, Y_train, Y_valid, Y_train_valid, Y_test = train_valid_test_split(X,Y, model_task)

In [ ]:
# save the uncorrected model, to then sort its features by importances
save_model=True
uncorrected_model_path = "/work/data/models/uncorrected_model.pkl"

Y_pred_train_valid = train_naive_model(X_train, X_valid, X_train_valid, X_test, Y_train, Y_valid, Y_train_valid, Y_test, model_task, stat_criteria, save_model=save_model,
                                    model_type=model_type)

In [ ]:
train_valid_set_with_uncorrected_results = augment_train_valid_set_with_results("uncorrected", X_train_valid, Y_train_valid, Y_pred_train_valid, model_task)
#train_valid_set_with_uncorrected_results

In [ ]:
train_valid_set_with_uncorrected_results.shape

In [ ]:
train_valid_set_with_uncorrected_results.to_csv('uncorrected_augmented_train_valid_set_label_encoded.csv')

Here, we observe the groups of features on which a gap is created by the model for "percentage positive" (i.e. demographic parity, a fairness metric unconditioned to the true labels):

In [ ]:
# we fix a minimum for injustice_acceptance
injustice_acceptance=3
min_individuals_discriminated=0.01

fairness_purpose="overall_positive_rate"

set_disadvantaging_features = discrimination_alert(train_valid_set_with_uncorrected_results,
                      "uncorrected", 
                      fairness_purpose, 
                      model_task, 
                      injustice_acceptance, 
                      min_individuals_discriminated)

set_disadvantaging_features

## Discrimination correction with a new fair model

We draw comparison between grid search and weights distorsion to generate fairer models, according to fairness purposes to maximize (and after evaluation of fairness metrics) conditional vs unconditional to true labels...

In [ ]:
list_protected_attributes = list({'age',
 'education',
 'marital-status',
 #'native-country',
 #'occupation',
 'relationship',
  'sex'})

#list_protected_attributes = ['sex']

compare_fairness_methods(
        X=X,
        Y=Y,
        train_valid_set_with_uncorrected_results=train_valid_set_with_uncorrected_results,
        list_protected_attributes=list_protected_attributes,
        model_type=model_type,
)